In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Import TorchSig components
from torchsig.datasets.datasets import TorchSigIterableDataset
from torchsig.geo.types import GeoPoint
from torchsig.geo.datasets import TorchSigGeoDataset, Transmitter, Receiver
from torchsig.geo.transforms import PathLoss, PathDelay
from torchsig.utils.defaults import TorchSigDefaults
from torchsig.transforms.transforms import Spectrogram

# Import example utility functions for visualization and geolocation
import matplotlib.pyplot as plt

from examples.scripts.geo_example_utils import (
    plot_geo_network_at_frame,
)

## `TorchSigGeoDataset` Config

In [ ]:
# Constants
NUM_SAMPLES = 4096
SAMPLE_RATE = 10_000_000 # 10 MHz
FFT_N = 512

# Modify default metadata
defaults = TorchSigDefaults()
gds_metadata = defaults.default_dataset_metadata.copy()

# Signal properties
gds_metadata["num_iq_samples_dataset"] = NUM_SAMPLES
gds_metadata["sample_rate"] = SAMPLE_RATE
gds_metadata["signal_center_freq_min"] = 250_000
gds_metadata["signal_center_freq_max"] = 750_000

# Spectrogram properties
gds_metadata["fft_size"] = FFT_N
gds_metadata["fft_stride"] = FFT_N
gds_metadata["frequency_min"] = -SAMPLE_RATE # for global antialiasing filter
gds_metadata["frequency_max"] = SAMPLE_RATE

## Define Geometry

In [ ]:
CENTER  = (37.7749, -122.4194, 100) # San Francisco

tx_pos  = GeoPoint(CENTER[0], CENTER[1], CENTER[2]) # centered

rx1_pos = GeoPoint(CENTER[0],       CENTER[1] - 0.1, CENTER[2] + 1000) # West
rx2_pos = GeoPoint(CENTER[0],       CENTER[1] + 0.1, CENTER[2] + 1000) # East
rx3_pos = GeoPoint(CENTER[0] - 0.1, CENTER[1],       CENTER[2] + 1000) # South

## `Transmitter` and `Receiver` Construction

In [ ]:
# Create RX
rx1 = Receiver(rx1_pos, SAMPLE_RATE, identifier="rx1")
rx2 = Receiver(rx2_pos, SAMPLE_RATE, identifier="rx2")
rx3 = Receiver(rx3_pos, SAMPLE_RATE, identifier="rx3")

# create TX
txds_metadata = gds_metadata.copy()
txds          = TorchSigIterableDataset(
                    metadata=txds_metadata,
                    signal_generators=["qpsk"],
                    seed=42)
tx = Transmitter(txds, tx_pos, identifier="tx")

## Plot the geometry

In [ ]:
from matplotlib.ticker import FormatStrFormatter

fig, ax = plt.subplots(1,1)
ax.xaxis.set_major_formatter(FormatStrFormatter('%.1f'))

plot_geo_network_at_frame([tx], 
                          [rx1, rx2, rx3], 
                          0, connections=True, ax=ax)

plt.show()

## Putting it all together

In [ ]:
gds = TorchSigGeoDataset(
    transmitters=[tx],
    receivers=[rx1,rx2,rx3],
    channel_transforms=[PathLoss(model="custom", loss_db=2), PathDelay()]
)

## Generating Samples

In [ ]:
s1 = next(gds)
s2 = next(gds)
s3 = next(gds)

## Plot received signals

In [ ]:
fig, ax = plt.subplots(3,constrained_layout=True)

for i,s in enumerate([s1,s2,s3]):
    
    # plot spectrogram
    rx_spectrogram = Spectrogram(fft_size=FFT_N)(s.copy())
    im    = ax[i].imshow(rx_spectrogram.data, aspect='auto', cmap='viridis')
    
    # pretty the labels
    ax[i].set_title(f"Receiver {s['rx_id']}", pad=10)
    ax[i].set_xlabel('Time')
    ax[i].set_ylabel('Frequency')

fig.colorbar(im, ax=ax)
plt.show()